My Code

In [27]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
import random

#-----CONFIG-----#
MODEL_NAME = "distilgpt2"
START_TEXT = "The weather today is "
NUM_STEPS = 30
K = 5
PICK_STRATEGY = "top_k" # "greedy" or top_k

In [28]:
#-----LOAD MODEL-----#
print("Loading AI model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print(f"\nStarting text: '{START_TEXT}'")
print(f"Steps: {NUM_STEPS}, k: {K}, strategy: {PICK_STRATEGY}")
print("\nGenerating token by token...\n")

#-----GENERATION LOOP-----#
current_text = START_TEXT

for step in range(NUM_STEPS):
    print(f"--- Step {step + 1} ---")
    print(f"Current text: '{current_text}'")

    # Encode current text
    input_ids = tokenizer.encode(current_text, return_tensors="pt")

    # Forward pass
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits

    # Logits → probabilities for NEXT token
    next_token_logits = logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim=0)

    # Top-k candidates
    top_probs, top_indices = torch.topk(next_token_probs, K, dim=0)

    print(f"Top {K} next-token predictions:")
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        token = tokenizer.decode([idx])
        print(f"  {i+1}. '{token}' ({prob.item()*100:.2f}%)")

    #----PICK STRATEGY----
    if PICK_STRATEGY == "greedy":
        chosen_token_id = top_indices[0]
        strategy_note = "Greedy (highest probability)"

    elif PICK_STRATEGY == "top_k":
        normalized_probs = top_probs / torch.sum(top_probs)
        chosen_index = torch.multinomial(normalized_probs, 1).item()
        chosen_token_id = top_indices[chosen_index]
        strategy_note = "Top-k sampling"

    else:
        raise ValueError("PICK_STRATEGY must be 'greedy' or 'top_k'")

    # Append chosen token
    chosen_token = tokenizer.decode([chosen_token_id])
    current_text += chosen_token

    print(f"✓ Chosen token: '{chosen_token}'")
    print(f"Strategy used: {strategy_note}")
    print(f"Updated text: '{current_text}'\n")

print("===================================")
print(f"Final generated text:\n'{current_text}'")

Loading AI model...

Starting text: 'The weather today is '
Steps: 30, k: 5, strategy: top_k

Generating token by token...

--- Step 1 ---
Current text: 'The weather today is '
Top 5 next-token predictions:
  1. 'iced' (45.60%)
  2. ' ' (18.60%)
  3. '!!!' (2.48%)
  4. '�' (2.00%)
  5. '????' (1.78%)
✓ Chosen token: 'iced'
Strategy used: Top-k sampling
Updated text: 'The weather today is iced'

--- Step 2 ---
Current text: 'The weather today is iced'
Top 5 next-token predictions:
  1. ' up' (13.14%)
  2. ' with' (11.31%)
  3. ' and' (8.39%)
  4. '.' (8.00%)
  5. ',' (7.97%)
✓ Chosen token: '.'
Strategy used: Top-k sampling
Updated text: 'The weather today is iced.'

--- Step 3 ---
Current text: 'The weather today is iced.'
Top 5 next-token predictions:
  1. '�' (21.13%)
  2. ' I' (8.85%)
  3. '
' (5.88%)
  4. ' It' (5.63%)
  5. ' The' (4.90%)
✓ Chosen token: '�'
Strategy used: Top-k sampling
Updated text: 'The weather today is iced.�'

--- Step 4 ---
Current text: 'The weather today is

Original Code

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch

# Load model
print("Loading AI model...")
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading AI model...


In [ ]:
# Simple example
text = "The weather today is "
print(f"\nStarting text: '{text}'")
print("\nGenerating word by word...\n")

# Generate 5 words, one at a time
current_text = text
for step in range(5):
    print(f"--- Step {step + 1} ---")
    print(f"Current: '{current_text}'")

    # Encode current text
    input_ids = tokenizer.encode(current_text, return_tensors="pt")

    # Get predictions
    with torch.no_grad():
        outputs = model(input_ids)
        predictions = outputs.logits

    # Get the predictions for the NEXT token
    next_token_logits = predictions[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits, dim=0)

    # Get top 5 predictions
    top_probs, top_indices = torch.topk(next_token_probs, 5)

    print("Top 5 next word predictions:")
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        word = tokenizer.decode([idx])
        print(f"  {i+1}. '{word}' ({prob.item()*100:.1f}% confident)")

    # Use the most likely word
    next_token_id = top_indices[0]
    next_word = tokenizer.decode([next_token_id])
    current_text += next_word

    print(f"✓ Chosen: '{next_word}'")
    print(f"New text: '{current_text}'\n")

print(f"\nFinal generated text: '{current_text}'")


Starting text: 'The weather today is '

Generating word by word...

--- Step 1 ---
Current: 'The weather today is '
Top 5 next word predictions:
  1. 'iced' (45.6% confident)
  2. ' ' (18.6% confident)
  3. '!!!' (2.5% confident)
  4. '�' (2.0% confident)
  5. '????' (1.8% confident)
✓ Chosen: 'iced'
New text: 'The weather today is iced'

--- Step 2 ---
Current: 'The weather today is iced'
Top 5 next word predictions:
  1. ' up' (13.1% confident)
  2. ' with' (11.3% confident)
  3. ' and' (8.4% confident)
  4. '.' (8.0% confident)
  5. ',' (8.0% confident)
✓ Chosen: ' up'
New text: 'The weather today is iced up'

--- Step 3 ---
Current: 'The weather today is iced up'
Top 5 next word predictions:
  1. ',' (15.7% confident)
  2. ' and' (14.9% confident)
  3. '.' (10.3% confident)
  4. ' by' (6.9% confident)
  5. ' to' (4.9% confident)
✓ Chosen: ','
New text: 'The weather today is iced up,'

--- Step 4 ---
Current: 'The weather today is iced up,'
Top 5 next word predictions:
  1. ' and' 